```{contents}
```

## Generative Adversarial Network (GAN)

---

### **Core Idea**

A **Generative Adversarial Network (GAN)** is a **two-network system** —
a *Generator (G)* and a *Discriminator (D)* — that **compete** with each other in a **game-like setup**.

* **Generator (G):** tries to create *fake data* that looks real
* **Discriminator (D):** tries to *distinguish real from fake data*

Over time:

* The **Generator improves** in creating realistic samples.
* The **Discriminator improves** in detecting fakes.
  Eventually, the Generator becomes so good that the Discriminator can no longer tell the difference.

This setup is called **adversarial training**.

---

### **Architecture**

![image.png](../images/gan.png)

```
Noise (z) ──► [ Generator G ] ──► Fake Data ──►
                                └──► [ Discriminator D ] ──► Real/Fake (0/1)
Real Data ─────────────────────►/
```

---

### **Mathematical Formulation**

The GAN objective is a **minimax game**:

$$
\min_G \max_D V(D, G) =
\mathbb{E}*{x \sim p*{data}(x)} [\log D(x)] +
\mathbb{E}_{z \sim p_z(z)} [\log(1 - D(G(z)))]
$$

Where:

* $D(x)$: probability that $x$ is real
* $G(z)$: generator output (fake sample)
* $p_{data}$: real data distribution
* $p_z$: noise distribution (usually Gaussian or uniform)

---

### **Step-by-Step Workflow**

#### **Input Noise**

* Start with random noise $z \sim N(0, 1)$
* This acts as the “seed” for generation

#### **Generator (G)**

* Takes noise $z$
* Produces fake data $G(z)$
* Learns to transform noise into realistic-looking samples

Mathematically:
$$
G: z \rightarrow x_{fake}
$$

#### **Discriminator (D)**

* Binary classifier: distinguishes *real data (1)* from *fake data (0)*
* Learns to maximize:
  $$
  \log D(x_{real}) + \log(1 - D(G(z)))
  $$

#### **Adversarial Training Loop**

1. Train **D** to correctly classify real vs fake
2. Train **G** to fool **D** (make fake data look real)
3. Alternate updates — the two networks improve against each other

---

### **Training Objective Simplified**

| Network               | Goal                            | Optimization                               |
| --------------------- | ------------------------------- | ------------------------------------------ |
| **Discriminator (D)** | Maximize correct classification | $\max_D [\log D(x) + \log(1 - D(G(z)))]$ |
| **Generator (G)**     | Fool the discriminator          | $\min_G [\log(1 - D(G(z)))]$             |

When both reach equilibrium → **Nash equilibrium**, GAN is said to be trained.

---

### **Intuitive Analogy**

Think of a **counterfeiter vs. detective**:

* 🧑‍🎨 **Generator = Counterfeiter:** tries to make fake currency that looks real.
* 👮 **Discriminator = Detective:** tries to detect counterfeit money.
* Over time, both improve — the counterfeiter gets better, and the detective gets sharper — until even the detective can’t tell the difference.

---

### **Types of GANs**

| Type                               | Description                                             | Use Case                             |
| ---------------------------------- | ------------------------------------------------------- | ------------------------------------ |
| **Vanilla GAN**                    | Basic GAN (as described above)                          | Simple data generation               |
| **DCGAN (Deep Convolutional GAN)** | Uses CNN layers                                         | Image generation                     |
| **Conditional GAN (cGAN)**         | Generator takes labels as input                         | Class-conditional image synthesis    |
| **CycleGAN**                       | Maps between two domains without paired data            | Style transfer (e.g., horse → zebra) |
| **Pix2Pix**                        | Paired image-to-image translation                       | Black→color image, edge→photo        |
| **WGAN (Wasserstein GAN)**         | Uses Wasserstein distance to improve training stability | High-quality stable generation       |
| **StyleGAN**                       | Generates photo-realistic human faces                   | Deepfake, art synthesis              |

---

### **Applications**

| Domain                             | Example                                                |
| ---------------------------------- | ------------------------------------------------------ |
| 🎨 **Image Generation**            | Generate faces, anime, artwork                         |
| 🖼️ **Image-to-Image Translation** | Sketch → Real image, Night → Day                       |
| 🧠 **Data Augmentation**           | Generate synthetic training data                       |
| 🧍 **Deepfake Creation**           | Face swapping, video synthesis                         |
| 🎶 **Audio Synthesis**             | Generate realistic speech or music                     |
| 🧾 **Text-to-Image**               | (DALL·E, Stable Diffusion use GAN/Transformer hybrids) |

---

### **Advantages**

* Generates **high-quality**, realistic data
* Learns **data distribution** implicitly (no need for explicit probability modeling)
* Useful for **unsupervised learning**

---

### **Disadvantages**

* **Training instability** (minimax game can oscillate or collapse)
* **Mode collapse:** Generator produces limited variety
* **Difficult convergence** — needs careful balancing of G and D learning rates
* **Hard to evaluate** quantitatively (no direct loss measure like accuracy)

---

### **Example: PyTorch Implementation (Simple DCGAN)**

```python
import torch
from torch import nn

# Generator
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(100, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Tanh()
        )

    def forward(self, z):
        return self.model(z)

# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
```

---

### **Training Flow**

1. Sample real images from dataset
2. Generate fake images from noise
3. Train discriminator (real → 1, fake → 0)
4. Train generator (to make D output 1 for fake samples)
5. Repeat

---

**Key Metric**

No fixed accuracy — we use:

* **FID (Fréchet Inception Distance):** measures visual similarity between real and generated samples
* **IS (Inception Score):** measures both diversity and realism

---

**Summary**

| Component             | Role                             | Output                    |
| --------------------- | -------------------------------- | ------------------------- |
| **Generator (G)**     | Creates fake samples             | $G(z)$                  |
| **Discriminator (D)** | Classifies real/fake             | Probability               |
| **Loss Function**     | Minimax (adversarial)            | $\min_G \max_D V(D, G)$ |
| **Goal**              | Reach equilibrium                | $D(G(z)) \approx 0.5$   |
| **Application**       | Synthetic data, image generation | High realism              |


```{dropdown} Click here for Sections
```{tableofcontents}